# DUK leg kinematics

Interactive inspection of the reusable `duk_core` four-bar model. The numerical solver uses a local A-D frame, while the notebook displays the configured robot/sketch frame where AD is oriented +30° from +Ox.


In [ ]:
from pathlib import Path
import math

from duk_core.modeling import load_leg_geometry_json, solve_leg_pose, leg_jacobian, pose_in_sketch_frame, sample_leg_trajectory
from duk_core.visualization import draw_leg_linkage, draw_point_trajectories, plot_point_coordinates, plot_point_jacobian

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'robot/config/physical/duk_leg_geometry.json').exists():
            return candidate
    raise FileNotFoundError('Run the notebook from a DUK repository checkout')

repo_root = find_repo_root()
geometry = load_leg_geometry_json(repo_root / 'robot/config/physical/duk_leg_geometry.json')
print(f'AD orientation = {math.degrees(geometry.ad_orientation_rad):.1f} deg from +Ox')
geometry


In [ ]:
lower, upper = geometry.geometric_angle_limits_rad()
print(f'BO = {geometry.bo_mm:.3f} mm')
print(f'CO = {geometry.co_mm:.3f} mm')
print(f'geometric angle range = {math.degrees(lower):.3f} .. {math.degrees(upper):.3f} deg')
print('The endpoints are singular boundaries, not operating limits.')


In [ ]:
branch = 1  # Calculation branch only; verify the physical CAD branch before treating this as robot truth.
local_pose = solve_leg_pose(geometry, math.radians(75.0), assembly_branch=branch)
robot_pose = pose_in_sketch_frame(geometry, local_pose)
local_jac = leg_jacobian(geometry, local_pose)
print('O in robot/sketch frame =', robot_pose.o, 'mm')
print('Local dO/dalpha =', local_jac.d_o, 'mm/rad')
draw_leg_linkage(robot_pose, geometry).set_title('DUK leg linkage — robot/sketch frame (AD = +30° from +Ox)')


In [ ]:
margin = math.radians(2.0)
local_poses = sample_leg_trajectory(geometry, lower + margin, upper - margin, 180, assembly_branch=branch)
robot_poses = tuple(pose_in_sketch_frame(geometry, pose) for pose in local_poses)
draw_point_trajectories(robot_poses).set_title('B/C/H/O trajectories — robot/sketch frame')


In [ ]:
plot_point_coordinates(robot_poses, 'O').set_title('Wheel-center coordinates — robot/sketch frame')


In [ ]:
plot_point_jacobian(geometry, local_poses, 'O', rotation_rad=geometry.ad_orientation_rad).set_title('Wheel-center Jacobian — robot/sketch frame')
